# Folder: **interconnections**

In [ ]:
######################################## Parameters

### List of market price files
mp_list = [
            'mp_FR_2030.csv',
            'mp_PT_2030.csv',
]

In [ ]:
##### Import packages
import yaml
import os 
import sys
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp
from functions.notebook_common import load_file_csv


##### Read params.yaml
with open('../params.yaml', 'r') as configfile:
    params = yaml.safe_load(configfile)


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `ic_market_prices`

Plot heatmaps of the market price used for PT and FR.

In [ ]:
######################################## Parameters

### Min and max values for representation. Leave empty for auto-scaling
vmin = ''
vmax = 200


In [ ]:

def plot_heatmap_from_csv(df, filename, vmin=None, vmax=None):
    # Verify that the DataFrame has 8760 values (365 days * 24 hours)
    if df.shape[0] != 8760:
        raise ValueError(f"File {filename} does not have 8760 values, it has {df.shape[0]} rows.")
    
    # Reshape data to have 365 rows (days) and 24 columns (hours)
    data_matrix = df.values.reshape(365, 24)
    
    # Use data min/max if vmin or vmax are empty
    if vmin == '' or vmin is None:
        vmin = data_matrix.min()
    if vmax == '' or vmax is None:
        vmax = data_matrix.max()

    # Create heatmap
    plt.figure(figsize=(10, 4))
    sns.heatmap(data_matrix.T, cmap='viridis', cbar_kws={'label': 'Value'}, vmin=vmin, vmax=vmax)
        
    # Row and column labels
    plt.title(filename)
    plt.ylabel('Hour of day')
    plt.xlabel('Day of year')    
    
    # Display plot
    plt.tight_layout()



for file in mp_list:
    df = load_file_csv(params, file, location='data_ES', prefix='interconnections', folder='ic_market_prices')
    if vmax not in ('', None):
        series_max = float(df.to_numpy().max())
        values_above_vmax = int((df.to_numpy().flatten() > vmax).sum())
        print(f"{file}: {values_above_vmax} values are above vmax ({vmax}). Maximum value in the series: {round(series_max,2)} EUR/MWh.")
    plot_heatmap_from_csv(df, file, vmin=vmin, vmax=vmax)


Plot violin diagrams of the market price used for PT and FR.

In [ ]:
######################################## Parameters

### Min and max values for representation. Leave empty for auto-scaling
vmin = ''
vmax = 200


In [ ]:
def plot_violin_all_files(data_dict, vmin=None, vmax=None):
    # List to accumulate all data
    all_values = []
    all_files = []
    
    # Read each file and store its data
    for filename, df in data_dict.items():
        
        if df.shape[0] != 8760:
            raise ValueError(f"File {filename} does not have 8760 values, it has {df.shape[0]} rows.")
        
        values = df.values.flatten()
        
        # Store values and corresponding filename
        all_values.extend(values)
        all_files.extend([filename] * len(values))
    
    # Use data min/max if vmin or vmax are empty
    if vmin == '' or vmin is None:
        vmin = min(all_values)
    if vmax == '' or vmax is None:
        vmax = max(all_values)
    
    # Create combined DataFrame
    violin_data = pd.DataFrame({
        'File': all_files,
        'Value': all_values
    })
    
    # Create violin diagram
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='File', y='Value', data=violin_data, palette='viridis')
    
    # Titles and labels
    plt.title("Violin plot of market price time series")
    plt.xlabel('')
    plt.ylabel('EUR/MWh')

    # Set Y-axis limits if specified
    if vmin is not None and vmin != '':
        plt.ylim(vmin, vmax)
    
    # Display plot
    plt.tight_layout()
    plt.show()



# Load files
data_dict = {}
for file in mp_list:
    df = load_file_csv(params, file, location='data_ES', prefix='interconnections', folder='ic_market_prices')
    if vmax not in ('', None):
        series_max = float(df.to_numpy().max())
        values_above_vmax = int((df.to_numpy().flatten() > vmax).sum())
        print(f"{file}: {values_above_vmax} values are above vmax ({vmax}). Maximum value in the series: {round(series_max,2)} EUR/MWh.")
    data_dict[file] = df


# Plot
plot_violin_all_files(data_dict, vmin=vmin, vmax=vmax)